In [1]:
import polars as pl
import polars.selectors as cs
from scipy.stats import norm
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import calendar

In [48]:
cpiu = (
    pl.read_excel("cpiu.xlsx", read_options={"header_row": 11})
    .rename({m: str(i) for i, m in enumerate(calendar.month_abbr[1:])})
    .unpivot(
        map(str, range(12)),
        index=["Year", "Annual"],
        variable_name="month",
        value_name="cpiu",
    )
    .sort("Year")
    .with_columns(
        pl.col("month").str.to_integer(),
        annual_computed=pl.col("cpiu").mean().over("Year"),
    )
    .with_columns(
        annual_delta=pl.when(
            pl.col("annual_computed").shift(-1) != pl.col("annual_computed")
        )
        .then(
            (pl.col("annual_computed").shift(-1) - pl.col("annual_computed"))
            / pl.col("annual_computed")
        )
        .backward_fill()
        .forward_fill()
    )
    .with_columns(
        filled_cpiu=pl.coalesce(
            pl.col("cpiu"),
            (1 + pl.col("annual_delta") / 12).pow(pl.col("month"))
            * pl.col("cpiu").first().over("Year"),
        )
    )
    .with_columns(
        eoy_delta=(pl.col("filled_cpiu") - pl.col("filled_cpiu").first())
        / pl.col("filled_cpiu").first()
    )
)

In [50]:
budget = (
    pl.concat(
        [
            pl.read_json("budget2024.json", infer_schema_length=None),
            pl.read_json("budget2025.json", infer_schema_length=None),
        ]
    )
    .unnest("data")
    .unnest("budgetVsActualV2")
)

In [63]:
pl.DataFrame({'a':[1,1,2,2],'b':[1,2,3,4]}).with_columns(c=pl.col('b').sample(1).implode().over('a'))

a,b,c
i64,i64,list[i64]
1,1,[1]
1,2,[1]
2,3,[3]
2,4,[3]


In [51]:
def extract_cashflow(cashflow: pl.Series):
    return (
        cashflow.explode()
        .struct.unnest()
        .select("subAccounts", headline_category="category")
        .explode("subAccounts")
        .unnest("subAccounts")
        .select(
            "headline_category",
            "category",
            pl.col("annual").struct.field("transactions"),
        )
        .explode("transactions")
        .unnest("transactions")
        .with_columns(pl.col("date").str.to_date())
        .drop(["transactionType", "__typename"])
        .drop_nulls()
    )


expenses = extract_cashflow(budget["expenses"])

cashflow = pl.concat(
    [
        extract_cashflow(budget["incomes"]).with_columns(is_income=pl.lit(True)),
        expenses.with_columns(is_income=pl.lit(False)),
    ]
)

# Simulate 13 months because we need to see what happens this December to know how much budget we start next year with.
MONTHS_TO_SIMULATE = 13
SIMULATIONS = 10000
UNPAID_BILLS = 8828.77
# Based on the "Operation" account pulled from Daisy dashboard on 2025-11-18
STARTING_BALANCE = 24844.00 - UNPAID_BILLS
# Simulate budget increases between 1% - 5% in increments of 1 percentage point.
BUDGET_INCREASES = list(
    round(budget_increase / 100.0, 2) for budget_increase in range(-5, 7)
)
cashflow_by_month_by_increase = [
    (
        budget_increase,
        cashflow.with_columns(
            pl.col("amount")
            * pl.when(pl.col("is_income")).then(1 + budget_increase).otherwise(-1)
        )
        .group_by(
            year=pl.col("date").dt.year(),
            month=pl.col("date").dt.month(),
        )
        .agg(pl.col("amount").sum()),
    )
    for budget_increase in BUDGET_INCREASES
]

simulations = pl.concat(
    pl.collect_all(
        (
            cashflow_by_month.lazy()
            .select(
                pl.col("amount").sample(MONTHS_TO_SIMULATE, shuffle=True).cum_sum()
                + pl.lit(STARTING_BALANCE),
            )
            .with_columns(
                simulation_id=pl.lit(simulation_id),
                budget_increase=pl.lit(budget_increase),
            )
            .with_row_index("month")
            for (
                budget_increase,
                cashflow_by_month,
            ) in cashflow_by_month_by_increase
            for simulation_id in range(0, SIMULATIONS)
        )
    )
)

In [52]:
budget_increases_series = pl.Series("budget_increase", BUDGET_INCREASES)

monthly_cashflow_by_budget_increase = (
    cashflow.join(budget_increases_series.to_frame(), how="cross")
    .with_columns(
        pl.col("amount")
        * pl.when(pl.col("is_income")).then(1 + pl.col("budget_increase")).otherwise(-1)
    )
    .group_by(
        "budget_increase",
        year=pl.col("date").dt.year(),
        month=pl.col("date").dt.month(),
    )
    .agg(pl.col("amount").sum())
    .sort("month")
    .pivot("month", index=["budget_increase", "year"], values="amount")
    .drop("year")
)
from functools import reduce


monthly_simulations = (
    reduce(
        lambda df, c: df.join(
            monthly_cashflow_by_budget_increase.select("budget_increase", c),
            on="budget_increase",
        ),
        monthly_cashflow_by_budget_increase.select(cs.digit()).columns,
        budget_increases_series.to_frame(),
    )
    .with_columns(pl.row_index("simulation_id").over("budget_increase"))
    .unpivot(
        cs.digit(),
        index=["budget_increase", "simulation_id"],
        variable_name="month",
        value_name="amount",
    )
    .sort("budget_increase", "simulation_id", pl.col("month").str.to_integer())
    .with_columns(
        pl.col("amount").cum_sum().over("budget_increase", "simulation_id")
        + STARTING_BALANCE
    )
)

In [ ]:
simulations = monthly_simulations

In [54]:
simulations_by_month = (
    simulations.group_by("month", "budget_increase")
    .agg(
        amount_average=pl.col("amount").mean(),
        amount_min=pl.col("amount").min(),
        amount_max=pl.col("amount").max(),
    )
    .sort("budget_increase", "month")
)

TRENDLINE_FIG_COL_COUNT = 2

fig = make_subplots(
    rows=int(len(BUDGET_INCREASES) / TRENDLINE_FIG_COL_COUNT),
    cols=TRENDLINE_FIG_COL_COUNT,
    shared_yaxes="all",
    subplot_titles=[
        f"budget_increase={budget_increase}" for budget_increase in BUDGET_INCREASES
    ],
)
for i, budget_increase in enumerate(BUDGET_INCREASES):
    sims = simulations_by_month.filter(pl.col("budget_increase") == budget_increase)
    row = int(i / TRENDLINE_FIG_COL_COUNT) + 1
    col = i % TRENDLINE_FIG_COL_COUNT + 1
    fig.add_trace(
        go.Scatter(
            x=pl.concat([sims["month"], sims["month"].reverse()]),
            y=pl.concat([sims["amount_min"], sims["amount_max"].reverse()]),
            name=f"min/max {budget_increase}",
            fill="toself",
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(x=sims["month"], y=sims["amount_average"], name=budget_increase),
        row=row,
        col=col,
    )
    fig.update_xaxes(title_text="month", row=row, col=col)
fig.update_layout(height=1000, legend=go.layout.Legend(title="budget_increase"))
fig.show()

In [55]:
simulation_mins = simulations.group_by("budget_increase", "simulation_id").agg(
    pl.col("amount").min()
)

In [56]:
fig = px.pie(
    simulation_mins.group_by(
        "budget_increase",
        ruinous=pl.col("amount") < 0,
    )
    .len("simulation_count")
    .with_columns(
        ruinous=pl.when("ruinous")
        .then(pl.lit("Special Assessment"))
        .otherwise(pl.lit("Safe"))
    )
    .sort("budget_increase"),
    names="ruinous",
    values="simulation_count",
    facet_col="budget_increase",
    facet_col_wrap=3,
    title="Likelihood of Special Assessment",
    color_discrete_sequence=["#4B08AF", "#32965D"],
    height=1000,
)
fig.show(renderer="notebook_connected")

# Special Assessment
95% confidence that the special assessment--if there is one--will be less than `amount`

In [57]:
# Inflation estimation from https://www.federalreserve.gov/monetarypolicy/files/fomcprojtabl20250917.pdf
INFLATION = 0.026
simulation_mins.filter(pl.col("amount") < 0).sort("budget_increase").with_columns(
    -pl.col("amount")
).group_by("budget_increase").agg(
    amount_95_conf=pl.col("amount").mean() + pl.col("amount").std() * norm.ppf(0.95),
    amount_99_conf=pl.col("amount").mean() + pl.col("amount").std() * norm.ppf(0.99),
).with_columns(
    amount_95_conf_with_inflation=pl.col("amount_95_conf") * (1 + INFLATION),
    amount_99_conf_with_inflation=pl.col("amount_99_conf") * (1 + INFLATION),
)

budget_increase,amount_95_conf,amount_99_conf,amount_95_conf_with_inflation,amount_99_conf_with_inflation
f64,f64,f64,f64,f64
-0.05,18681.289366,23133.710782,19167.00289,23735.187262
-0.04,17152.612316,21349.901284,17598.580237,21904.998718
-0.03,16240.286025,20071.75511,16662.533461,20593.620743
-0.02,14996.7289,18499.553199,15386.643851,18980.541582
-0.01,13581.036673,16814.333879,13934.143627,17251.50656
…,…,…,…,…
0.02,10005.052198,12434.11748,10265.183555,12757.404534
0.03,8729.648129,10911.200367,8956.61898,11194.891576
0.04,7597.103235,9535.95399,7794.627919,9783.888794


In [58]:
fig = px.bar(
    expenses.with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="headline_category",
)
fig.show(renderer="notebook_connected")

In [59]:
fig = px.bar(
    expenses.with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="category",
)
fig.show(renderer="notebook_connected")